In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# LOAD DATA & FIT FULL MLR MODEL
# ─────────────────────────────────────────────
df = pd.read_csv(r"C:\Users\drisy\Downloads\warehouse_order_processing.csv")

X = df.drop(columns=["Order_Processing_Time_min"])
y = df["Order_Processing_Time_min"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train_c = sm.add_constant(X_train)
model     = sm.OLS(y_train, X_train_c).fit()

# ─────────────────────────────────────────────
# WHAT IS T-TEST IN REGRESSION?
# ─────────────────────────────────────────────
print("=" * 65)
print("         T-TEST FOR REGRESSION COEFFICIENTS")
print("=" * 65)
print("""
PURPOSE:
  The t-test in MLR checks whether each individual predictor
  has a statistically significant relationship with the
  response variable (Processing Time), holding all other
  predictors constant.

HYPOTHESIS (for each predictor Xj):
  H0: βj = 0  → predictor has NO effect on Y
  H1: βj ≠ 0  → predictor HAS a significant effect on Y

DECISION RULE:
  If |t-statistic| > t-critical  →  Reject H0
  If p-value < 0.05              →  Reject H0 (significant)
  If p-value ≥ 0.05              →  Fail to reject H0 (not significant)
""")

# ─────────────────────────────────────────────
# T-TEST RESULTS FOR EACH PREDICTOR
# ─────────────────────────────────────────────
print("=" * 65)
print("T-TEST RESULTS — EACH PREDICTOR")
print("=" * 65)

# Degrees of freedom for t-critical
df_resid   = model.df_resid
alpha      = 0.05
t_critical = stats.t.ppf(1 - alpha/2, df=df_resid)  # two-tailed

print(f"\n  Sample size (train)     : {len(y_train)}")
print(f"  Number of predictors    : {X.shape[1]}")
print(f"  Degrees of freedom      : {int(df_resid)}")
print(f"  Alpha (significance)    : {alpha}")
print(f"  t-critical (two-tailed) : ±{t_critical:.4f}\n")

ttest_results = []
predictors    = X.columns.tolist()

for pred in predictors:
    coef   = model.params[pred]
    se     = model.bse[pred]
    t_stat = model.tvalues[pred]
    p_val  = model.pvalues[pred]
    ci_low, ci_high = model.conf_int().loc[pred]

    reject   = abs(t_stat) > t_critical
    decision = "Reject H0 ✓" if reject else "Fail to Reject H0 ✗"
    sig      = "Significant" if reject else "Not Significant"

    ttest_results.append({
        "Predictor"  : pred,
        "Coefficient": round(coef,   4),
        "Std Error"  : round(se,     4),
        "t-Statistic": round(t_stat, 4),
        "p-value"    : round(p_val,  6),
        "CI Lower"   : round(ci_low, 4),
        "CI Upper"   : round(ci_high,4),
        "Decision"   : decision,
        "Result"     : sig
    })

    print(f"  Predictor : {pred}")
    print(f"    Coefficient  : {coef:.4f}")
    print(f"    Std Error    : {se:.4f}")
    print(f"    t-Statistic  : {t_stat:.4f}")
    print(f"    p-value      : {p_val:.4e}")
    print(f"    95% CI       : [{ci_low:.4f}, {ci_high:.4f}]")
    print(f"    Decision     : {decision} — {sig}")
    print()

# ─────────────────────────────────────────────
# SUMMARY TABLE
# ─────────────────────────────────────────────
print("=" * 65)
print("SUMMARY TABLE")
print("=" * 65)
ttest_df = pd.DataFrame(ttest_results)
print(ttest_df[["Predictor","Coefficient","t-Statistic",
                "p-value","Decision","Result"]].to_string(index=False))

sig_count   = sum(1 for r in ttest_results if r["Result"] == "Significant")
insig_count = len(ttest_results) - sig_count
print(f"\n  Significant predictors     : {sig_count} / {len(ttest_results)}")
print(f"  Non-significant predictors : {insig_count} / {len(ttest_results)}")

# Save to CSV
ttest_df.to_csv("ttest_results.csv", index=False)
print("  Saved as 'ttest_results.csv'")

# ─────────────────────────────────────────────
# VISUALIZATION 1: t-Statistics with critical line
# ─────────────────────────────────────────────
ttest_df_sorted = ttest_df.sort_values("t-Statistic")

colors = ['#4CAF50' if abs(t) > t_critical else '#F44336'
          for t in ttest_df_sorted["t-Statistic"]]

plt.figure(figsize=(10, 6))
bars = plt.barh(ttest_df_sorted["Predictor"],
                ttest_df_sorted["t-Statistic"],
                color=colors, edgecolor='white')
plt.axvline( t_critical, color='black', lw=2, linestyle='--',
             label=f'+t-critical = {t_critical:.2f}')
plt.axvline(-t_critical, color='black', lw=2, linestyle='--',
             label=f'-t-critical = -{t_critical:.2f}')
plt.axvline(0, color='gray', lw=1)
plt.xlabel("t-Statistic", fontsize=11)
plt.title("T-Test: t-Statistics for Each Predictor\n(Green = Significant | Red = Not Significant)",
          fontsize=12, fontweight='bold')
plt.legend(fontsize=9)
plt.tight_layout()
plt.savefig("ttest_tstatistics.png", dpi=150, bbox_inches='tight')
plt.close()
print("\n  Saved as 'ttest_tstatistics.png'")

# ─────────────────────────────────────────────
# VISUALIZATION 2: Coefficient plot with 95% CI
# ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
y_pos = np.arange(len(predictors))

for i, row in ttest_df.iterrows():
    color = '#4CAF50' if row["Result"] == "Significant" else '#F44336'
    ax.plot([row["CI Lower"], row["CI Upper"]], [i, i],
            color=color, lw=3, solid_capstyle='round')
    ax.scatter(row["Coefficient"], i, color=color, s=80, zorder=5)

ax.axvline(0, color='black', lw=1.5, linestyle='--', label='Zero line')
ax.set_yticks(range(len(ttest_df)))
ax.set_yticklabels(ttest_df["Predictor"])
ax.set_xlabel("Coefficient Value", fontsize=11)
ax.set_title("95% Confidence Intervals for Regression Coefficients\n(Green = Significant | Red = Not Significant)",
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("ttest_confidence_intervals.png", dpi=150, bbox_inches='tight')
plt.close()
print("  Saved as 'ttest_confidence_intervals.png'")

# ─────────────────────────────────────────────
# VISUALIZATION 3: p-value plot with alpha line
# ─────────────────────────────────────────────
pval_df = ttest_df.sort_values("p-value", ascending=False)
colors  = ['#4CAF50' if p < 0.05 else '#F44336' for p in pval_df["p-value"]]

plt.figure(figsize=(10, 6))
plt.barh(pval_df["Predictor"], pval_df["p-value"],
         color=colors, edgecolor='white')
plt.axvline(0.05, color='black', lw=2, linestyle='--',
            label='α = 0.05 (significance threshold)')
plt.xlabel("p-value", fontsize=11)
plt.title("T-Test: p-values for Each Predictor\n(Green = Significant (p<0.05) | Red = Not Significant)",
          fontsize=12, fontweight='bold')
plt.legend(fontsize=9)
plt.tight_layout()
plt.savefig("ttest_pvalues.png", dpi=150, bbox_inches='tight')
plt.close()
print("  Saved as 'ttest_pvalues.png'")

# ─────────────────────────────────────────────
# INTERCEPT T-TEST
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("T-TEST FOR INTERCEPT (β0)")
print("=" * 65)
intercept_t = model.tvalues['const']
intercept_p = model.pvalues['const']
intercept_c = model.params['const']
print(f"  Intercept (β0) : {intercept_c:.4f}")
print(f"  t-Statistic    : {intercept_t:.4f}")
print(f"  p-value        : {intercept_p:.4e}")
print(f"  Decision       : {'Reject H0 — Intercept is significant ✓' if intercept_p < 0.05 else 'Fail to Reject H0 ✗'}")

print("\n" + "=" * 65)
print("ALL DONE! Output files:")
print("  ttest_results.csv")
print("  ttest_tstatistics.png")
print("  ttest_confidence_intervals.png")
print("  ttest_pvalues.png")
print("=" * 65)

         T-TEST FOR REGRESSION COEFFICIENTS

PURPOSE:
  The t-test in MLR checks whether each individual predictor
  has a statistically significant relationship with the
  response variable (Processing Time), holding all other
  predictors constant.

HYPOTHESIS (for each predictor Xj):
  H0: βj = 0  → predictor has NO effect on Y
  H1: βj ≠ 0  → predictor HAS a significant effect on Y

DECISION RULE:
  If |t-statistic| > t-critical  →  Reject H0
  If p-value < 0.05              →  Reject H0 (significant)
  If p-value ≥ 0.05              →  Fail to reject H0 (not significant)

T-TEST RESULTS — EACH PREDICTOR

  Sample size (train)     : 800
  Number of predictors    : 8
  Degrees of freedom      : 791
  Alpha (significance)    : 0.05
  t-critical (two-tailed) : ±1.9630

  Predictor : Num_Items_Ordered
    Coefficient  : 0.8035
    Std Error    : 0.0155
    t-Statistic  : 51.7233
    p-value      : 5.2567e-256
    95% CI       : [0.7730, 0.8340]
    Decision     : Reject H0 ✓ — Signific